In [ ]:
%pip install pandas -q
%pip install matplotlib -q
%pip install scipy -q

In [ ]:
from google.colab import drive
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import scipy

drive.mount('/content/drive')

%matplotlib inline

In [ ]:
tag_relation_mapping = {
    'Ekosistem': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Karasal Ekosistem', 'Yerleşim Yerleri', 'Sucul Ekosistem'], 'siblings': []},
    'Karasal Ekosistem': {'type': 'middle', 'hypernyms': ['Ekosistem'], 'hyponyms': ['Yerleşim Yerleri'], 'siblings': ['Sucul Ekosistem']},
    'Yerleşim Yerleri': {'type': 'bottom', 'hypernyms': ['Ekosistem', 'Karasal Ekosistem'], 'hyponyms': [], 'siblings': []},
    'Sucul Ekosistem': {'type': 'bottom', 'hypernyms': ['Ekosistem'], 'hyponyms': [], 'siblings': ['Karasal Ekosistem']},
    'Kirletici': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Sıvı Kirletici', 'Katı Kirletici', 'Gaz Kirletici',  'Enerji'], 'siblings': []},
    'Sıvı Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Katı Kirletici', 'Gaz Kirletici', 'Enerji']},
    'Katı Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Gaz Kirletici', 'Enerji']},
    'Gaz Kirletici': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Katı Kirletici', 'Enerji']},
    'Enerji': {'type': 'bottom', 'hypernyms': ['Kirletici'], 'hyponyms': [], 'siblings': ['Sıvı Kirletici', 'Katı Kirletici', 'Gaz Kirletici']},
    'Afet': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Doğal Afet', 'İnsan Kaynaklı Afet'], 'siblings': []},
    'Doğal Afet': {'type': 'bottom', 'hypernyms': ['Afet'], 'hyponyms': [], 'siblings': ['İnsan Kaynaklı Afet']},
    'İnsan Kaynaklı Afet': {'type': 'bottom', 'hypernyms': ['Afet'], 'hyponyms': [], 'siblings': ['Doğal Afet']},
    'Biota': {'type': 'top', 'hypernyms': [], 'hyponyms': ['İnsan Dışı Biota', 'Sucul Biota', 'Karasal Biota', 'İnsan'], 'siblings': []},
    'İnsan Dışı Biota': {'type': 'middle', 'hypernyms': ['Biota'], 'hyponyms': ['Sucul Biota', 'Karasal Biota'], 'siblings': ['İnsan']},
    'Sucul Biota': {'type': 'bottom', 'hypernyms': ['İnsan Dışı Biota', 'Biota'], 'hyponyms': [], 'siblings': ['Karasal Biota']},
    'Karasal Biota': {'type': 'bottom', 'hypernyms': ['İnsan Dışı Biota', 'Biota'], 'hyponyms': [], 'siblings': ['Sucul Biota']},
    'İnsan': {'type': 'bottom', 'hypernyms': ['Biota'], 'hyponyms': [], 'siblings': ['İnsan Dışı Biota']},
    'Çevresel Etki': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Ekolojik Etki', 'Refah Etkisi', 'Ekonomik Etki', 'Sağlık Etkisi', 'Sosyal Etki'], 'siblings': []},
    'Ekolojik Etki': {'type': 'bottom', 'hypernyms': ['Çevresel Etki'], 'hyponyms': [], 'siblings': ['Refah Etkisi']},
    'Refah Etkisi': {'type': 'middle', 'hypernyms': ['Çevresel Etki'], 'hyponyms': ['Ekonomik Etki', 'Sağlık Etkisi', 'Sosyal Etki'], 'siblings': ['Ekolojik Etki']},
    'Ekonomik Etki': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Sağlık Etkisi',  'Sosyal Etki']},
    'Sağlık Etkisi': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Ekonomik Etki',  'Sosyal Etki']},
    'Sosyal Etki': {'type': 'bottom', 'hypernyms': ['Refah Etkisi', 'Çevresel Etki'], 'hyponyms': [], 'siblings': ['Ekonomik Etki',  'Sağlık Etkisi']},
    'Çevre Yönetimi': {'type': 'top', 'hypernyms': [], 'hyponyms': ['Düzenleme', 'Azaltma', 'Arıtım'], 'siblings': []},
    'Düzenleme': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Azaltma', 'Arıtım']},
    'Azaltma': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Düzenleme', 'Arıtım']},
    'Arıtım': {'type': 'bottom', 'hypernyms': ['Çevre Yönetimi'], 'hyponyms': [], 'siblings': ['Düzenleme', 'Azaltma']},
    'Kirleten': {'type': 'top', 'hypernyms': [], 'hyponyms': ['İnsan Kaynaklı Kirleten',  'Doğal Kirleten'], 'siblings': []},
    'İnsan Kaynaklı Kirleten': {'type': 'bottom', 'hypernyms': ['Kirleten'], 'hyponyms': [], 'siblings': ['Doğal Kirleten']},
    'Doğal Kirleten': {'type': 'bottom', 'hypernyms': ['Kirleten'], 'hyponyms': [], 'siblings': ['İnsan Kaynaklı Kirleten']}
}

# labels with no hypernyms
top_level_labels = []
# labels that have hypernyms and hyponyms
mid_level_labels = []
# labels with no hyponyms
bottom_level_labels = []
# labels with siblings
sibling_labels = []
# labels without siblings
no_sibling_labels = []

for tag in tag_relation_mapping:
  if len(tag_relation_mapping[tag]['hypernyms']) == 0:
    top_level_labels.append(tag)
  elif len(tag_relation_mapping[tag]['hyponyms']) == 0:
    bottom_level_labels.append(tag)
  else:
    mid_level_labels.append(tag)

  if len(tag_relation_mapping[tag]['siblings']) > 0:
    sibling_labels.append(tag)
  else:
    no_sibling_labels.append(tag)

In [ ]:
def clear_result(result):
  result = result.replace("[", "")
  result = result.replace("]", "")
  return result

def trim_result(row, result_name, average_name):
  column = row[f'{result_name}_avg_{average_name}'].split(" ")
  column = map(clear_result, column)
  column = list(filter(lambda x: x != '', column))
  if len(column) == 1:
    column = float(column[0])
  elif len(column) == 2:
    column = float(column[1])

  return column

def trim_results(row, average_name):
  precision = trim_result(row, 'precision', average_name)
  recall = trim_result(row, 'recall', average_name)
  f1 = trim_result(row, 'f1', average_name)
  row['average_type'] = average_name
  row['precision'] = precision
  row['recall'] = recall
  row['f1'] = f1

  hidden_labels = row['hidden_labels'].replace("[", '')
  hidden_labels = hidden_labels.replace("]", '')
  hidden_labels= hidden_labels.replace("'", "")
  hidden_labels = hidden_labels.split(",")

  row['hidden_labels'] = list(filter(lambda x : x != '', hidden_labels))
  row['hidden_labels_count'] = len(row['hidden_labels'])
  return row

In [ ]:
# Dataset abbreviations
# DN => Distilbert model with only news dataset
# DNG => Distilbert model with news and gpt dataset
# TN => Turkish NER model with only news dataset
# TNG => Turkish NER model with news and gpt dataset

#df_DN = pd.read_csv('/content/drive/MyDrive/ner_results/distilbert_turkish_cased_news_data_final.csv')
#df_TN = pd.read_csv('/content/drive/MyDrive/ner_results/turkish_ner_news_data_final.csv')
#df_DNG = pd.read_csv('/content/drive/MyDrive/ner_results/distilbert-base-turkish-cased_news_and_gpt_data_final.csv')
#df_TNG = pd.read_csv('/content/drive/MyDrive/ner_results/turkish_ner_news_and_gpt_data_final.csv')

# df_DN = pd.read_csv('/content/drive/MyDrive/ner_results/distilbert-base-turkish-cased_17-07-2024-18-13_seed_117_version_101_news_data.csv')
# df_TN = pd.read_csv('/content/drive/MyDrive/ner_results/distilbert-base-turkish-cased_17-07-2024-17-25_seed_117_version_101_news_and_gpt_data.csv')
# df_DNG = pd.read_csv('/content/drive/MyDrive/ner_results/turkish_ner_17-07-2024-19-30_seed_117_version_101_news_data.csv')
# df_TNG = pd.read_csv('/content/drive/MyDrive/ner_results/turkish_ner_17-07-2024-19-21_seed_117_version_101_news_and_gpt_data.csv')

df_DN = pd.read_csv('/content/drive/MyDrive/ner_results/july_v2/distilbert-base-turkish-cased_version_101_news_only_without_validation.csv')
df_TN = pd.read_csv('/content/drive/MyDrive/ner_results/july_v2/turkish_ner_version_101_news_only_without_validation.csv')
df_DNG = pd.read_csv('/content/drive/MyDrive/ner_results/july_v2/distilbert-base-turkish-cased_version_101_ner_and_gpt_data_without_validation.csv')
df_TNG = pd.read_csv('/content/drive/MyDrive/ner_results/july_v2/turkish_ner_version_101_news_and_gpt_data_without_validation.csv')

df_DN['enum'] = 'DN'
df_TN['enum'] = 'TN'
df_DNG['enum'] = 'DNG'
df_TNG['enum'] = 'TNG'

result_names = [
    'precision_avg',
    'recall_avg',
    'f1_avg'
]

remove_result_averages = [
    'none',
    'binary',
    'macro',
    'weighted',
    'micro'
]

df_remove_results = []
for result_name in result_names:
  for remove_result_average in remove_result_averages:
    df_remove_results.append(f'{result_name}_{remove_result_average}')

df_remove_columns = [
    'model_checkpoint',
    'tokenizer_checkpoint',
    'hyponyms',
    'hypernyms',
    'siblings',
    'train_runtime',
    'train_steps_per_second',
    'train_loss',
    'train_length',
    'validation_length',
    'total_length',
    'precision',
    'recall',
    'f1_var',
    'average_type',
    'accuracy',
    'matrix'
]

df_remove_columns.extend(df_remove_results)

In [ ]:
def prepare_dataframe(df):
  df = df.apply(trim_results, axis=1, args=('none',))
  return df.drop(columns=df_remove_columns, axis=1)

df_DN

In [ ]:
df_DN = prepare_dataframe(df_DN)
df_TN = prepare_dataframe(df_TN)
df_DNG = prepare_dataframe(df_DNG)
df_TNG = prepare_dataframe(df_TNG)

In [ ]:
df_DN

In [ ]:
all_df = {}
df_ALL = pd.concat([df_DN, df_TN, df_DNG, df_TNG])
df_ALL

In [ ]:
def merge_sets_apply(df):
  DN = df[df['enum'] == 'DN']
  TN = df[df['enum'] == 'TN']
  DNG = df[df['enum'] == 'DNG']
  TNG = df[df['enum'] == 'TNG']

  if DNG.empty:
    return None

  series =  pd.Series({
      'class_unseen': DNG['class_unseen'].iloc[0],
      'shot_number': DNG['shot_number'].iloc[0],
      'hidden_labels': DNG['hidden_labels'].iloc[0],
      'removed_hyponyms': DNG['removed_hyponyms'].iloc[0],
      'removed_hypernyms': DNG['removed_hypernyms'].iloc[0],
      'removed_siblings': DNG['removed_siblings'].iloc[0],
      'hidden_labels_count': DNG['hidden_labels_count'].iloc[0],
      'N_test_length': int(DN['test_length'].iloc[0]) if not DN.empty else np.NaN,
      'NG_test_length': DNG['test_length'].iloc[0],
      'DN_f1': DN['f1'].iloc[0] if not DN.empty else np.NaN,
      'TN_f1': TN['f1'].iloc[0] if not DN.empty else np.NaN,
      'DNG_f1': DNG['f1'].iloc[0],
      'TNG_f1': TNG['f1'].iloc[0]
  })

  series['DN_f1'] = series['DN_f1'] if series['DN_f1'] != 0 else np.NaN
  series['TN_f1'] = series['TN_f1'] if series['TN_f1'] != 0 else np.NaN
  series['DNG_f1'] = series['DNG_f1'] if series['DNG_f1'] != 0 else np.NaN
  series['TNG_f1'] = series['TNG_f1'] if series['TNG_f1'] != 0 else np.NaN

  if pd.isna(series['DN_f1']) or pd.isna(series['TN_f1']) or pd.isna(series['DNG_f1']) or pd.isna(series['TNG_f1']):
    series['has_any_none'] = True
  else:
    series['has_any_none'] = False

  return series


df_ALL = df_ALL.groupby(['class_unseen', 'shot_number', 'removed_hyponyms', 'removed_siblings', 'hidden_labels_count']).apply(merge_sets_apply).reset_index(drop=True, inplace=False)

In [ ]:
df_ALL

In [ ]:
def add_delta(row):
  if pd.isna(row['DN_f1']) or pd.isna(row['TN_f1']):
    row['DN_TN_DELTA_F1'] = np.NaN
    row['DN_DNG_DELTA_F1'] = np.NaN
    row['TN_TNG_DELTA_F1'] = np.NaN
  else:
    row['DN_TN_DELTA_F1'] = row['TN_f1'] - row['DN_f1']
    row['DN_DNG_DELTA_F1'] = row['DNG_f1'] - row['DN_f1']
    row['TN_TNG_DELTA_F1'] = row['TNG_f1'] - row['TN_f1']

  row['DNG_TNG_DELTA_F1'] = row['TNG_f1'] - row['DNG_f1']
  return row


df_ALL = df_ALL.apply(add_delta, axis=1)
all_df['ALL'] = df_ALL




In [ ]:
df_ALL

In [ ]:
df_ZERO_SHOT = df_ALL[df_ALL['shot_number'] == 0]
all_df['ZERO_SHOT'] = df_ZERO_SHOT
df_ZERO_SHOT

In [ ]:
df_ONE_SHOT = df_ALL[df_ALL['shot_number'] == 1]
all_df['ONE_SHOT'] = df_ONE_SHOT
df_ONE_SHOT

In [ ]:
df_TEN_SHOT = df_ALL[df_ALL['shot_number'] == 10]
all_df['TEN_SHOT'] = df_TEN_SHOT
df_TEN_SHOT

In [ ]:
df_TOP_RESULTS = df_ALL[df_ALL['class_unseen'].isin(top_level_labels)]
df_TOP_RESULTS

In [ ]:
df_REMOVED_HYPONYMS = df_TOP_RESULTS[df_TOP_RESULTS['removed_hyponyms']]
all_df['REMOVED_HYPONYMS'] = df_REMOVED_HYPONYMS
df_REMOVED_HYPONYMS_ZERO_SHOT = df_REMOVED_HYPONYMS[df_REMOVED_HYPONYMS['shot_number'] == 0]
df_REMOVED_HYPONYMS_ONE_SHOT = df_REMOVED_HYPONYMS[df_REMOVED_HYPONYMS['shot_number'] == 1]
df_REMOVED_HYPONYMS_TEN_SHOT = df_REMOVED_HYPONYMS[df_REMOVED_HYPONYMS['shot_number'] == 10]
all_df['REMOVED_HYPONYMS_ZERO_SHOT'] = df_REMOVED_HYPONYMS_ZERO_SHOT
all_df['REMOVED_HYPONYMS_ONE_SHOT'] = df_REMOVED_HYPONYMS_ONE_SHOT
all_df['REMOVED_HYPONYMS_TEN_SHOT'] = df_REMOVED_HYPONYMS_TEN_SHOT
df_REMOVED_HYPONYMS

In [ ]:
df_WITH_HYPONYMS = df_TOP_RESULTS[~df_TOP_RESULTS['removed_hyponyms']]
all_df['WITH_HYPONYMS'] = df_WITH_HYPONYMS
df_WITH_HYPONYMS_ZERO_SHOT = df_WITH_HYPONYMS[df_WITH_HYPONYMS['shot_number'] == 0]
df_WITH_HYPONYMS_ONE_SHOT = df_WITH_HYPONYMS[df_WITH_HYPONYMS['shot_number'] == 1]
df_WITH_HYPONYMS_TEN_SHOT = df_WITH_HYPONYMS[df_WITH_HYPONYMS['shot_number'] == 10]
all_df['WITH_HYPONYMS_ZERO_SHOT'] = df_WITH_HYPONYMS_ZERO_SHOT
all_df['WITH_HYPONYMS_ONE_SHOT'] = df_WITH_HYPONYMS_ONE_SHOT
all_df['WITH_HYPONYMS_TEN_SHOT'] = df_WITH_HYPONYMS_TEN_SHOT
df_WITH_HYPONYMS

In [ ]:
df_BOTTOM_RESULTS = df_ALL[df_ALL['class_unseen'].isin(bottom_level_labels)]
df_BOTTOM_RESULTS['has_siblings'] = df_BOTTOM_RESULTS['class_unseen'].apply(lambda x: x in sibling_labels)
df_BOTTOM_RESULTS = df_BOTTOM_RESULTS[(df_BOTTOM_RESULTS['has_siblings'] & ~df_BOTTOM_RESULTS['removed_siblings']) | (~df_BOTTOM_RESULTS['has_siblings'])]
df_BOTTOM_RESULTS = df_BOTTOM_RESULTS.drop(['has_siblings'], axis=1)
df_BOTTOM_RESULTS

In [ ]:
df_WITH_HYPERNYMS = df_BOTTOM_RESULTS[~df_BOTTOM_RESULTS['removed_hypernyms']]
all_df['WITH_HYPERNYMS'] = df_WITH_HYPERNYMS
df_WITH_HYPERNYMS_ZERO_SHOT = df_WITH_HYPERNYMS[df_WITH_HYPERNYMS['shot_number'] == 0]
df_WITH_HYPERNYMS_ONE_SHOT = df_WITH_HYPERNYMS[df_WITH_HYPERNYMS['shot_number'] == 1]
df_WITH_HYPERNYMS_TEN_SHOT = df_WITH_HYPERNYMS[df_WITH_HYPERNYMS['shot_number'] == 10]
all_df['WITH_HYPERNYMS_ZERO_SHOT'] = df_WITH_HYPERNYMS_ZERO_SHOT
all_df['WITH_HYPERNYMS_ONE_SHOT'] = df_WITH_HYPERNYMS_ONE_SHOT
all_df['WITH_HYPERNYMS_TEN_SHOT'] = df_WITH_HYPERNYMS_TEN_SHOT
df_WITH_HYPERNYMS

In [ ]:
df_WITHOUT_HYPERNYMS = df_BOTTOM_RESULTS[df_BOTTOM_RESULTS['removed_hypernyms']]
all_df['WITHOUT_HYPERNYMS'] = df_WITHOUT_HYPERNYMS
df_WITHOUT_HYPERNYMS_ZERO_SHOT = df_WITHOUT_HYPERNYMS[df_WITHOUT_HYPERNYMS['shot_number'] == 0]
df_WITHOUT_HYPERNYMS_ONE_SHOT = df_WITHOUT_HYPERNYMS[df_WITHOUT_HYPERNYMS['shot_number'] == 1]
df_WITHOUT_HYPERNYMS_TEN_SHOT = df_WITHOUT_HYPERNYMS[df_WITHOUT_HYPERNYMS['shot_number'] == 10]
all_df['WITHOUT_HYPERNYMS_ZERO_SHOT'] = df_WITHOUT_HYPERNYMS_ZERO_SHOT
all_df['WITHOUT_HYPERNYMS_ONE_SHOT'] = df_WITHOUT_HYPERNYMS_ONE_SHOT
all_df['WITHOUT_HYPERNYMS_TEN_SHOT'] = df_WITHOUT_HYPERNYMS_TEN_SHOT
df_WITHOUT_HYPERNYMS

In [ ]:
df_MIDDLE_RESULTS = df_ALL[df_ALL['class_unseen'].isin(mid_level_labels)]
df_MIDDLE_RESULTS = df_MIDDLE_RESULTS[~df_MIDDLE_RESULTS['removed_siblings']]
df_MIDDLE_RESULTS

In [ ]:
df_WITH_RELATIVES = df_MIDDLE_RESULTS[(~df_MIDDLE_RESULTS['removed_hyponyms']) & (~df_MIDDLE_RESULTS['removed_hypernyms'])]
all_df['WITH_RELATIVES'] = df_WITH_RELATIVES
df_WITH_RELATIVES_ZERO_SHOT = df_WITH_RELATIVES[df_WITH_RELATIVES['shot_number'] == 0]
df_WITH_RELATIVES_ONE_SHOT = df_WITH_RELATIVES[df_WITH_RELATIVES['shot_number'] == 1]
df_WITH_RELATIVES_TEN_SHOT = df_WITH_RELATIVES[df_WITH_RELATIVES['shot_number'] == 10]
all_df['WITH_RELATIVES_ZERO_SHOT'] = df_WITH_RELATIVES_ZERO_SHOT
all_df['WITH_RELATIVES_ONE_SHOT'] = df_WITH_RELATIVES_ONE_SHOT
all_df['WITH_RELATIVES_TEN_SHOT'] = df_WITH_RELATIVES_TEN_SHOT
df_WITH_RELATIVES

In [ ]:
df_WITHOUT_RELATIVES = df_MIDDLE_RESULTS[(df_MIDDLE_RESULTS['removed_hyponyms']) & (df_MIDDLE_RESULTS['removed_hypernyms'])]
all_df['WITHOUT_RELATIVES'] = df_WITHOUT_RELATIVES
df_WITHOUT_RELATIVES_ZERO_SHOT = df_WITHOUT_RELATIVES[df_WITHOUT_RELATIVES['shot_number'] == 0]
df_WITHOUT_RELATIVES_ONE_SHOT = df_WITHOUT_RELATIVES[df_WITHOUT_RELATIVES['shot_number'] == 1]
df_WITHOUT_RELATIVES_TEN_SHOT = df_WITHOUT_RELATIVES[df_WITHOUT_RELATIVES['shot_number'] == 10]
all_df['WITHOUT_RELATIVES_ZERO_SHOT'] = df_WITHOUT_RELATIVES_ZERO_SHOT
all_df['WITHOUT_RELATIVES_ONE_SHOT'] = df_WITHOUT_RELATIVES_ONE_SHOT
all_df['WITHOUT_RELATIVES_TEN_SHOT'] = df_WITHOUT_RELATIVES_TEN_SHOT
df_WITHOUT_RELATIVES

In [ ]:
df_SIBLING_RESULTS = df_ALL[df_ALL['class_unseen'].isin(sibling_labels) &
                            ((df_ALL['class_unseen'].isin(top_level_labels) & ~df_ALL['removed_hyponyms'])
                            | (df_ALL['class_unseen'].isin(bottom_level_labels) & ~df_ALL['removed_hypernyms'])
                            | (df_ALL['class_unseen'].isin(mid_level_labels) & ~df_ALL['removed_hypernyms'] & ~df_ALL['removed_hyponyms']))]
df_SIBLING_RESULTS

In [ ]:
df_WITHOUT_SIBLINGS = df_SIBLING_RESULTS[df_SIBLING_RESULTS['removed_siblings']]
all_df['WITHOUT_SIBLINGS'] = df_WITHOUT_SIBLINGS
df_WITHOUT_SIBLINGS_ZERO_SHOT = df_WITHOUT_SIBLINGS[df_WITHOUT_SIBLINGS['shot_number'] == 0]
df_WITHOUT_SIBLINGS_ONE_SHOT = df_WITHOUT_SIBLINGS[df_WITHOUT_SIBLINGS['shot_number'] == 1]
df_WITHOUT_SIBLINGS_TEN_SHOT = df_WITHOUT_SIBLINGS[df_WITHOUT_SIBLINGS['shot_number'] == 10]
all_df['WITHOUT_SIBLINGS_ZERO_SHOT'] = df_WITHOUT_SIBLINGS_ZERO_SHOT
all_df['WITHOUT_SIBLINGS_ONE_SHOT'] = df_WITHOUT_SIBLINGS_ONE_SHOT
all_df['WITHOUT_SIBLINGS_TEN_SHOT'] = df_WITHOUT_SIBLINGS_TEN_SHOT
df_WITHOUT_SIBLINGS

In [ ]:
df_WITH_SIBLINGS = df_SIBLING_RESULTS[~df_SIBLING_RESULTS['removed_siblings']]
all_df['WITH_SIBLINGS'] = df_WITH_SIBLINGS
df_WITH_SIBLINGS_ZERO_SHOT = df_WITH_SIBLINGS[df_WITH_SIBLINGS['shot_number'] == 0]
df_WITH_SIBLINGS_ONE_SHOT = df_WITH_SIBLINGS[df_WITH_SIBLINGS['shot_number'] == 1]
df_WITH_SIBLINGS_TEN_SHOT = df_WITH_SIBLINGS[df_WITH_SIBLINGS['shot_number'] == 10]
all_df['WITH_SIBLINGS_ZERO_SHOT'] = df_WITH_SIBLINGS_ZERO_SHOT
all_df['WITH_SIBLINGS_ONE_SHOT'] = df_WITH_SIBLINGS_ONE_SHOT
all_df['WITH_SIBLINGS_TEN_SHOT'] = df_WITH_SIBLINGS_TEN_SHOT
df_WITH_SIBLINGS

In [ ]:
df_ALL.describe()

In [ ]:
def t_test(df, column1, column2):
  df = df.dropna(subset=[column1, column2])
  t_stat, p_val = scipy.stats.mstats.ttest_rel(df[column1], df[column2], alternative='greater')
  return t_stat, p_val

In [ ]:
df_T_TEST = pd.DataFrame(columns=['RESULT', 'TN-DN-t_stat', 'TN-DN-p_val', 'DNG_DN-t_stat', 'DNG_DN-p_val', 'TNG_TN-t_stat', 'TNG_TN-p_val', 'TNG_DNG-t_stat', 'TNG_DNG-p_val'])

def t_test_all_combinations(name, df):
  df_result = {'RESULT': name}
  df_result['TN-DN-t_stat'], df_result['TN-DN-p_val'] = t_test(df, 'TN_f1', 'DN_f1')
  df_result['DNG_DN-t_stat'], df_result['DNG_DN-p_val'] = t_test(df, 'DNG_f1', 'DN_f1')
  df_result['TNG_TN-t_stat'], df_result['TNG_TN-p_val'] = t_test(df, 'TNG_f1', 'TN_f1')
  df_result['TNG_DNG-t_stat'], df_result['TNG_DNG-p_val'] = t_test(df, 'TNG_f1', 'DNG_f1')

  return df_T_TEST._append(df_result, ignore_index=True)


In [ ]:
for name in all_df:
  df_T_TEST = t_test_all_combinations(name, all_df[name])

df_T_TEST
